# 🧠 DenseNet-40 GỐC (Original Replica)
**Kiến trúc**: DenseNet-40 (L=40, k=12) | **Bám sát 100% bài báo gốc** | KHÔNG có nâng cấp

### So sánh với bản nâng cấp (DoAnML)
| Thành phần | Bản gốc (repo này) | Bản nâng cấp (DoAnML) |
|---|---|---|
| Hàm kích hoạt | ReLU | Mish |
| Attention | Không có | SE Block |
| Class name | `DenseNetOriginal` | `DenseNetCustom` |

### Cấu trúc dự án
| File | Chức năng |
|------|----------|
| `model.py` | Kiến trúc DenseNet GỐC (DenseLayer, DenseBlock, TransitionLayer, DenseNetOriginal) |
| `data_loader.py` | Tải và xử lý dữ liệu CIFAR-10, CIFAR-100, SVHN |
| `train.py` | Hàm huấn luyện, đánh giá, lưu checkpoint |
| `utils.py` | Vẽ biểu đồ Loss/Accuracy |

## 1. Cài đặt Môi trường (Dùng chung)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

project_folder = '/content/drive/MyDrive/DenseNet_Original'
if not os.path.exists(project_folder):
    os.makedirs(project_folder)

%cd {project_folder}

In [ ]:
import os
import sys

GITHUB_USERNAME = "daq1209"
REPO_NAME = "OriginalDenseNetPythonReplica"
REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
PROJECT_ROOT = "/content/drive/MyDrive/DenseNet_Original"

DATA_DIR = f"{PROJECT_ROOT}/data"
RESULTS_DIR = f"{PROJECT_ROOT}/results"

if not os.path.exists(PROJECT_ROOT):
    os.makedirs(PROJECT_ROOT)
%cd {PROJECT_ROOT}

if os.path.exists(REPO_NAME):
    print(f"--- Đang làm mới thư mục {REPO_NAME} ---")
    !rm -rf {REPO_NAME}

print(f"--- Đang clone {REPO_NAME} từ {REPO_URL} ---")
!git clone {REPO_URL}

working_dir = os.path.abspath(REPO_NAME)
if working_dir not in sys.path:
    sys.path.append(working_dir)

%cd {working_dir}

print("\n--- Kiểm tra các file module ---")
required_files = ['model.py', 'train.py', 'data_loader.py', 'utils.py']
for f in required_files:
    status = "✅" if os.path.exists(f) else "❌"
    print(f"{status} {f}")

if os.path.exists("requirements.txt"):
    !pip install -r requirements.txt

import torch
if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
else:
    print("\n⚠️ Chưa bật GPU!")

---
## 2. Huấn luyện trên CIFAR-10 (10 lớp)

### 2.1. Tải dữ liệu CIFAR-10

In [ ]:
from data_loader import get_data_loaders, DATASET_NUM_CLASSES

DATASET_C10 = "cifar10"
trainloader_c10, testloader_c10 = get_data_loaders(
    dataset_name=DATASET_C10, data_dir=DATA_DIR
)

### 2.2. Khởi tạo mô hình GỐC (num_classes=10)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from model import DenseNetOriginal

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_epochs_c10 = 50

model_c10 = DenseNetOriginal(
    growth_rate=12,
    block_config=(12, 12, 12),
    num_classes=DATASET_NUM_CLASSES[DATASET_C10]
)
model_c10 = model_c10.to(device)

total_params = sum(p.numel() for p in model_c10.parameters())
print(f"Tổng số tham số (Bản GỐC): {total_params:,}")

criterion_c10 = nn.CrossEntropyLoss()
optimizer_c10 = optim.Adam(model_c10.parameters(), lr=0.001)
scheduler_c10 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_c10, T_max=num_epochs_c10)

print(f"✅ Đã khởi tạo mô hình GỐC [{DATASET_C10.upper()}] trên: {device}")

### 2.3. Huấn luyện CIFAR-10 (50 epochs)

In [ ]:
from train import train_one_epoch, evaluate, save_checkpoint

c10_train_losses, c10_train_accuracies = [], []
c10_test_losses, c10_test_accuracies = [], []

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN (Bản GỐC) [{DATASET_C10.upper()}]...")
for epoch in range(num_epochs_c10):
    print(f"\nEpoch {epoch+1}/{num_epochs_c10}")
    print("-" * 20)

    train_loss, train_acc = train_one_epoch(model_c10, trainloader_c10, optimizer_c10, criterion_c10, device)
    test_loss, test_acc = evaluate(model_c10, testloader_c10, criterion_c10, device)
    scheduler_c10.step()

    c10_train_losses.append(train_loss)
    c10_train_accuracies.append(train_acc)
    c10_test_losses.append(test_loss)
    c10_test_accuracies.append(test_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Test  Loss: {test_loss:.4f}  | Test  Acc: {test_acc:.2f}%")
    print(f"Learning Rate: {scheduler_c10.get_last_lr()[0]:.6f}")

    if (epoch + 1) % 10 == 0:
        save_checkpoint(model_c10, optimizer_c10, scheduler_c10, epoch + 1,
                        dataset_name=DATASET_C10, base_path=RESULTS_DIR)

print(f"\n🎉 HOÀN TẤT HUẤN LUYỆN (Bản GỐC) [{DATASET_C10.upper()}]!")

### 2.4. Trực quan hóa kết quả CIFAR-10

In [ ]:
from utils import plot_training_history
plot_training_history(c10_train_losses, c10_test_losses, c10_train_accuracies, c10_test_accuracies)

---
## 3. Huấn luyện trên CIFAR-100 (100 lớp)

### 3.1. Tải dữ liệu CIFAR-100

In [ ]:
from data_loader import get_data_loaders, DATASET_NUM_CLASSES

DATASET_C100 = "cifar100"
trainloader_c100, testloader_c100 = get_data_loaders(
    dataset_name=DATASET_C100, data_dir=DATA_DIR
)

### 3.2. Khởi tạo mô hình GỐC (num_classes=100)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from model import DenseNetOriginal

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_epochs_c100 = 50

model_c100 = DenseNetOriginal(
    growth_rate=12,
    block_config=(12, 12, 12),
    num_classes=DATASET_NUM_CLASSES[DATASET_C100]
)
model_c100 = model_c100.to(device)

total_params = sum(p.numel() for p in model_c100.parameters())
print(f"Tổng số tham số (Bản GỐC): {total_params:,}")

criterion_c100 = nn.CrossEntropyLoss()
optimizer_c100 = optim.Adam(model_c100.parameters(), lr=0.001)
scheduler_c100 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_c100, T_max=num_epochs_c100)

print(f"✅ Đã khởi tạo mô hình GỐC [{DATASET_C100.upper()}] trên: {device}")

### 3.3. Huấn luyện CIFAR-100 (50 epochs)

In [ ]:
from train import train_one_epoch, evaluate, save_checkpoint

c100_train_losses, c100_train_accuracies = [], []
c100_test_losses, c100_test_accuracies = [], []

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN (Bản GỐC) [{DATASET_C100.upper()}]...")
for epoch in range(num_epochs_c100):
    print(f"\nEpoch {epoch+1}/{num_epochs_c100}")
    print("-" * 20)

    train_loss, train_acc = train_one_epoch(model_c100, trainloader_c100, optimizer_c100, criterion_c100, device)
    test_loss, test_acc = evaluate(model_c100, testloader_c100, criterion_c100, device)
    scheduler_c100.step()

    c100_train_losses.append(train_loss)
    c100_train_accuracies.append(train_acc)
    c100_test_losses.append(test_loss)
    c100_test_accuracies.append(test_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Test  Loss: {test_loss:.4f}  | Test  Acc: {test_acc:.2f}%")
    print(f"Learning Rate: {scheduler_c100.get_last_lr()[0]:.6f}")

    if (epoch + 1) % 10 == 0:
        save_checkpoint(model_c100, optimizer_c100, scheduler_c100, epoch + 1,
                        dataset_name=DATASET_C100, base_path=RESULTS_DIR)

print(f"\n🎉 HOÀN TẤT HUẤN LUYỆN (Bản GỐC) [{DATASET_C100.upper()}]!")

### 3.4. Trực quan hóa kết quả CIFAR-100

In [ ]:
from utils import plot_training_history
plot_training_history(c100_train_losses, c100_test_losses, c100_train_accuracies, c100_test_accuracies)

---
## 4. Huấn luyện trên SVHN (10 lớp)

### 4.1. Tải dữ liệu SVHN

In [ ]:
from data_loader import get_data_loaders, DATASET_NUM_CLASSES

DATASET_SVHN = "svhn"
trainloader_svhn, testloader_svhn = get_data_loaders(
    dataset_name=DATASET_SVHN, data_dir=DATA_DIR
)

### 4.2. Khởi tạo mô hình GỐC (num_classes=10)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from model import DenseNetOriginal

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_epochs_svhn = 50

model_svhn = DenseNetOriginal(
    growth_rate=12,
    block_config=(12, 12, 12),
    num_classes=DATASET_NUM_CLASSES[DATASET_SVHN]
)
model_svhn = model_svhn.to(device)

total_params = sum(p.numel() for p in model_svhn.parameters())
print(f"Tổng số tham số (Bản GỐC): {total_params:,}")

criterion_svhn = nn.CrossEntropyLoss()
optimizer_svhn = optim.Adam(model_svhn.parameters(), lr=0.001)
scheduler_svhn = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_svhn, T_max=num_epochs_svhn)

print(f"✅ Đã khởi tạo mô hình GỐC [{DATASET_SVHN.upper()}] trên: {device}")

### 4.3. Huấn luyện SVHN (50 epochs)

In [ ]:
from train import train_one_epoch, evaluate, save_checkpoint

svhn_train_losses, svhn_train_accuracies = [], []
svhn_test_losses, svhn_test_accuracies = [], []

print(f"🚀 BẮT ĐẦU HUẤN LUYỆN (Bản GỐC) [{DATASET_SVHN.upper()}]...")
for epoch in range(num_epochs_svhn):
    print(f"\nEpoch {epoch+1}/{num_epochs_svhn}")
    print("-" * 20)

    train_loss, train_acc = train_one_epoch(model_svhn, trainloader_svhn, optimizer_svhn, criterion_svhn, device)
    test_loss, test_acc = evaluate(model_svhn, testloader_svhn, criterion_svhn, device)
    scheduler_svhn.step()

    svhn_train_losses.append(train_loss)
    svhn_train_accuracies.append(train_acc)
    svhn_test_losses.append(test_loss)
    svhn_test_accuracies.append(test_acc)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Test  Loss: {test_loss:.4f}  | Test  Acc: {test_acc:.2f}%")
    print(f"Learning Rate: {scheduler_svhn.get_last_lr()[0]:.6f}")

    if (epoch + 1) % 10 == 0:
        save_checkpoint(model_svhn, optimizer_svhn, scheduler_svhn, epoch + 1,
                        dataset_name=DATASET_SVHN, base_path=RESULTS_DIR)

print(f"\n🎉 HOÀN TẤT HUẤN LUYỆN (Bản GỐC) [{DATASET_SVHN.upper()}]!")

### 4.4. Trực quan hóa kết quả SVHN

In [ ]:
from utils import plot_training_history
plot_training_history(svhn_train_losses, svhn_test_losses, svhn_train_accuracies, svhn_test_accuracies)